# ⚡ Parallelization in LangGraph: Fan-Out and Fan-In

## Learning Objectives
In this notebook, you will learn:
1. **The sequential baseline** - A linear graph, a reducer, and why wall time is the sum of the steps
2. **Fan-out and fan-in** - Run independent nodes in the same step and join them, with a reducer so no write is lost
3. **Asymmetric parallel paths** - Join branches of different lengths with an explicit `add_edge([...], ...)` sync point
4. **Multi-source retrieval** - Search Wikipedia and the web in parallel, then synthesize one answer with an LLM

## Where this fits
Prompt chaining (module 05) ran steps one after another; routing (module 06) ran exactly one of several. Parallelization runs **all** of a fixed set of branches at once. This notebook covers the graph mechanics; `02_parallelization_sectioning_and_voting.ipynb` applies them to the pattern's two LLM variants, and `03_parallelization_lcel.ipynb` rebuilds this notebook in LCEL.

## Prerequisites
- LangGraph `StateGraph`, edges and state reducers — `modules/05_prompt_chaining/notebooks/01_prompt_chaining_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env` (Tavily is only used in Part 4; Wikipedia needs no key)

---
## 📖 What is Parallelization?

In this notebook, we'll explore one of the most powerful features of LangGraph: **parallel execution of nodes**. This capability is essential for building efficient AI agents that can perform multiple tasks simultaneously, significantly reducing overall execution time.

### Why Parallel Execution Matters

In real-world AI applications, agents often need to gather information from multiple sources, process data through different pipelines, or execute independent tasks. Without parallelization, these operations would run sequentially, leading to unnecessary delays.

**Example Use Cases:**
- Searching multiple knowledge bases simultaneously (Wikipedia + Web Search)
- Running multiple LLM calls in parallel for different subtasks
- Fetching data from multiple APIs at once
- Executing independent validation checks concurrently

| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), "Parallelization" section |

### Key Concepts:

LangGraph achieves parallelization through two fundamental mechanisms:

| Mechanism | Description |
|-----------|-------------|
| **Fan-Out** | A single node branches out to multiple nodes, allowing simultaneous execution of tasks |
| **Fan-In** | Multiple nodes converge back into a single node, aggregating results from parallel tasks |

---
## 📦 Step 1: Environment Setup

Load the API keys, then create the chat model (used only by the answer step in Part 4) through the course factory.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: Synthesizes the final answer in Part 4
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


---

## 🐌 Part 1: Simple Linear AI Workflow (Sequential Execution)

Before understanding parallel execution, let's first establish a baseline with a **sequential workflow**. In this pattern, each step runs one after another in a fixed order:

```
START → A → B → C → D → END
```

### Key Concepts:

1. **State Schema with TypedDict**: Defines what data flows through the graph
2. **Reducers (operator.add)**: Determines how state updates are combined
3. **Node Functions**: The actual logic that runs at each step
4. **Edges**: Define the flow/order of execution

Each node sleeps half a second to stand in for real work, so the wall time shows what actually ran in parallel.

In [3]:
# ============================================================================
# IMPORTS
# ============================================================================
import operator
import time
from typing import Annotated, Any

from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

STEP_SECONDS = 0.5  # simulated work per node, so wall time shows what ran in parallel

# ============================================================================
# STEP 1: DEFINE THE STATE SCHEMA
# ============================================================================
# The State class defines the structure of data that flows through our graph.
# Think of it as the "memory" or "context" that each node can read and write to.

class State(TypedDict):
    """
    State schema for our workflow.
    
    The `Annotated[list, operator.add]` syntax is crucial:
    - `list` is the data type
    - `operator.add` is the REDUCER function that defines HOW updates are merged
    
    When operator.add is applied to lists, it performs LIST CONCATENATION.
    This means: [1, 2] + [3] = [1, 2, 3]
    
    Without a reducer, the last update would OVERWRITE previous values!
    """
    state: Annotated[list, operator.add]

# ============================================================================
# STEP 2: DEFINE NODE LOGIC (Using a Callable Class)
# ============================================================================
# Nodes can be defined as functions OR callable classes.
# Using a class allows us to store configuration (like node_secret) during initialization.

class ReturnNodeValue:
    """
    A callable class that represents a graph node.
    
    When LangGraph executes this node, it calls the __call__ method.
    Each instance stores a unique identifier (node_secret) to track execution order,
    and sleeps STEP_SECONDS to stand in for real work (an API call, an LLM call).
    """
    
    def __init__(self, node_secret: str):
        """Store the node's identifier during initialization."""
        self._value = node_secret

    def __call__(self, state: State) -> Any:
        """
        This method is called when the node executes.
        
        Args:
            state: The current graph state (passed automatically by LangGraph)
            
        Returns:
            A dictionary with state updates. Keys match State fields.
            The value [self._value] is wrapped in a list because our reducer
            uses operator.add (list concatenation).
        """
        time.sleep(STEP_SECONDS)
        print(f"Adding {self._value} to {state['state']}")
        return {"state": [self._value]}

# ============================================================================
# STEP 3: BUILD THE GRAPH
# ============================================================================
# StateGraph is the main class for constructing LangGraph workflows.
# We pass the State class so the graph knows our state schema.

builder = StateGraph(State)

# ============================================================================
# STEP 4: ADD NODES TO THE GRAPH
# ============================================================================
# Each node has a unique name (string) and a callable (function or class instance).
# The names "a", "b", "c", "d" are used to reference nodes when adding edges.

builder.add_node("a", ReturnNodeValue("step A"))
builder.add_node("b", ReturnNodeValue("step B"))
builder.add_node("c", ReturnNodeValue("step C"))
builder.add_node("d", ReturnNodeValue("step D"))

# ============================================================================
# STEP 5: DEFINE THE FLOW (EDGES)
# ============================================================================
# Edges define the execution order. In a linear workflow, each node leads to exactly one other node.
# START and END are special constants representing entry and exit points.

builder.add_edge(START, "a")  # Entry point: START → a
builder.add_edge("a", "b")    # Sequential: a → b
builder.add_edge("b", "c")    # Sequential: b → c
builder.add_edge("c", "d")    # Sequential: c → d
builder.add_edge("d", END)    # Exit point: d → END

# ============================================================================
# STEP 6: COMPILE THE GRAPH
# ============================================================================
# compile() validates the graph structure and creates an executable workflow.
# After compilation, the graph is ready to process inputs.

graph = builder.compile()

print("✅ Linear graph compiled:", [n for n in graph.get_graph().nodes if not n.startswith("__")])

✅ Linear graph compiled: ['a', 'b', 'c', 'd']


### Execute the Linear Workflow

Let's run the graph and observe the execution order. Notice how:
1. Each step executes sequentially (A → B → C → D)
2. The state list grows with each step due to the `operator.add` reducer
3. The final state contains all steps in order: `["step A", "step B", "step C", "step D"]`

In [4]:
# ============================================================================
# RUN: Linear workflow
# ============================================================================
# Invoke the graph with an empty initial state
# The graph.invoke() method:
#   - Takes initial state as input
#   - Executes all nodes according to the defined edges
#   - Returns the final state after all nodes have executed

start = time.perf_counter()
result = graph.invoke({"state": []})
elapsed = time.perf_counter() - start

print("\n" + "="*50)
print("FINAL STATE:")
print("="*50)
print(f"Execution order: {result['state']}")
print(f"⏱️  Wall time: {elapsed:.2f}s (4 nodes x {STEP_SECONDS}s, one after another)")

Adding step A to []
Adding step B to ['step A']
Adding step C to ['step A', 'step B']
Adding step D to ['step A', 'step B', 'step C']

FINAL STATE:
Execution order: ['step A', 'step B', 'step C', 'step D']
⏱️  Wall time: 2.01s (4 nodes x 0.5s, one after another)


---

## 🔀 Part 2: Parallel Workflow with Fan-Out and Fan-In

Now let's introduce **parallel execution**! Instead of running all steps sequentially, we'll run some steps concurrently:

```
        ┌──→ B ──┐
START → A       ├──→ D → END
        └──→ C ──┘
```

### Key Concepts

| Concept | Description | Real-World Analogy |
|---------|-------------|-------------------|
| **Fan-Out** | One node branches to multiple nodes for simultaneous execution | Like a manager delegating tasks to multiple team members |
| **Fan-In** | Multiple nodes converge back to a single node | Like collecting results from all team members before moving forward |

### Why Use Parallel Execution?

1. **Performance**: Tasks B and C run simultaneously, reducing total execution time
2. **Independence**: B and C don't depend on each other's results
3. **Aggregation**: Node D receives combined results from both B and C

### Critical: Reducers in Parallel Workflows

When multiple nodes write to the **same state key** simultaneously, you MUST use a reducer to define how updates are combined:

```python
# Without reducer: Last write wins (data loss!)
state: list  # ❌ Dangerous in parallel workflows

# With reducer: Updates are merged correctly
state: Annotated[list, operator.add]  # ✅ Safe - lists are concatenated
```

> **Remember**: `operator.add` on lists performs concatenation: `[B] + [C] = [B, C]`

In [5]:
# ============================================================================
# PARALLEL EXECUTION WITH FAN-OUT AND FAN-IN
# ============================================================================
# State schema with reducer (same as before - essential for parallel writes)
class State(TypedDict):
    """
    State with operator.add reducer.
    This is CRITICAL for parallel execution - without it, 
    concurrent writes would overwrite each other!
    """
    state: Annotated[list, operator.add]

# ============================================================================
# BUILD THE PARALLEL GRAPH
# ============================================================================
builder = StateGraph(State)

# Add the same nodes as before
builder.add_node("a", ReturnNodeValue("step A"))
builder.add_node("b", ReturnNodeValue("step B"))
builder.add_node("c", ReturnNodeValue("step C"))
builder.add_node("d", ReturnNodeValue("step D"))

# ============================================================================
# DEFINE PARALLEL FLOW
# ============================================================================
# The key difference from linear workflow is in how we define edges:

# Entry point (same as linear)
builder.add_edge(START, "a")

# FAN-OUT: Node 'a' has TWO outgoing edges!
# This tells LangGraph to execute 'b' and 'c' IN PARALLEL
builder.add_edge("a", "b")  # a → b (parallel branch 1) 
builder.add_edge("a", "c")  # a → c (parallel branch 2)

# FAN-IN: Both 'b' and 'c' point to 'd'
# LangGraph will WAIT for BOTH to complete before running 'd'
builder.add_edge("b", "d")  # b → d (converge from branch 1)
builder.add_edge("c", "d")  # c → d (converge from branch 2)

# Exit point
builder.add_edge("d", END)

# Compile and visualize
graph = builder.compile()

print("✅ Parallel graph compiled — 'b' and 'c' branch from 'a' and converge at 'd'")

✅ Parallel graph compiled — 'b' and 'c' branch from 'a' and converge at 'd'


In [6]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = graph.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

🧩 Nodes: ['a', 'b', 'c', 'd']
🔗 Edges:
            __start__ ──▶ a                  (fixed)
                    a ──▶ b                  (fixed)
                    a ──▶ c                  (fixed)
                    b ──▶ d                  (fixed)
                    c ──▶ d                  (fixed)
                    d ──▶ __end__            (fixed)


In [7]:
# ============================================================================
# EXECUTE THE PARALLEL WORKFLOW
# ============================================================================
# Run the graph and observe:
# 1. Node 'a' executes first
# 2. Nodes 'b' and 'c' execute IN PARALLEL (order may vary!)
# 3. Node 'd' executes after BOTH 'b' and 'c' complete
# 4. The reducer combines outputs from parallel nodes

start = time.perf_counter()
result = graph.invoke({"state": []})
elapsed = time.perf_counter() - start

print("\n" + "="*50)
print("FINAL STATE:")
print("="*50)
print(f"Execution order: {result['state']}")
print(f"⏱️  Wall time: {elapsed:.2f}s — B and C overlapped, so ~3 steps, not 4")
print("\nNote: 'step B' and 'step C' ran in parallel - their order may vary!")

Adding step A to []
Adding step B to ['step A']Adding step C to ['step A']

Adding step D to ['step A', 'step B', 'step C']

FINAL STATE:
Execution order: ['step A', 'step B', 'step C', 'step D']
⏱️  Wall time: 1.51s — B and C overlapped, so ~3 steps, not 4

Note: 'step B' and 'step C' ran in parallel - their order may vary!


### Understanding the Output

Observe the execution:
- Node `b` and `c` both ran and appended their values to the state
- Thanks to the `operator.add` reducer, both values are preserved (not overwritten)
- The final state contains: `["step A", "step B", "step C", "step D"]`

> **Key Insight**: Without the reducer, whichever node finished last would overwrite the other's value, causing data loss!

---

## 🪜 Part 3: Asymmetric Parallel Paths

In real-world scenarios, parallel paths often have **different lengths**. For example:
- One path might involve a simple lookup
- Another path might require multiple processing steps

```
        ┌──→ B → X ──┐
START → A            ├──→ D → END
        └──→ C ──────┘
```

### The Challenge

When paths have different lengths, LangGraph needs to know when to proceed to the next node. 

### The Solution: List-Based Fan-In

Instead of adding separate edges, we use a **list syntax** to specify multiple source nodes:

```python
builder.add_edge(["x", "c"], "d")  # Wait for BOTH x AND c before running d
```

This explicitly tells LangGraph: "Don't run node `d` until both `x` and `c` have completed."

In [8]:
# ============================================================================
# ASYMMETRIC PARALLEL PATHS
# ============================================================================
# This example demonstrates handling parallel paths of DIFFERENT LENGTHS:
#   - Path 1: A → B → X (2 nodes after A)
#   - Path 2: A → C     (1 node after A)
# Both paths must complete before D can execute.

builder = StateGraph(State)

# ============================================================================
# ADD NODES
# ============================================================================
# Notice we now have 5 nodes: A, B, X, C, D
# Path 1 has an extra step (X) compared to Path 2

builder.add_node("a", ReturnNodeValue("step A"))  # Entry point
builder.add_node("b", ReturnNodeValue("step B"))  # Path 1, step 1
builder.add_node("x", ReturnNodeValue("step X"))  # Path 1, step 2 (extra step)
builder.add_node("c", ReturnNodeValue("step C"))  # Path 2, step 1
builder.add_node("d", ReturnNodeValue("step D"))  # Convergence point

# ============================================================================
# DEFINE ASYMMETRIC FLOW
# ============================================================================

# Entry point
builder.add_edge(START, "a")

# FAN-OUT: Split into two parallel paths
builder.add_edge("a", "b")  # Path 1 starts
builder.add_edge("a", "c")  # Path 2 starts

# Path 1 continues with an extra step
builder.add_edge("b", "x")  # B → X (Path 1 is longer)

# FAN-IN WITH LIST SYNTAX: Wait for BOTH paths to complete
# This is the KEY difference - using a list to specify multiple source nodes
# LangGraph will wait for BOTH "x" AND "c" before running "d"
builder.add_edge(["x", "c"], "d")  # Explicit synchronization point

# Exit point
builder.add_edge("d", END)

# Compile and visualize
graph = builder.compile()

print("✅ Asymmetric graph compiled — path A→B→X is longer than path A→C")

✅ Asymmetric graph compiled — path A→B→X is longer than path A→C


### Execution Behavior

In this configuration:
- Nodes `b` and `c` run in parallel after `a` completes
- Node `x` runs after `b` completes (sequential within Path 1)
- Node `d` **waits for both** `x` AND `c` to complete before running

This pattern is useful when:
- Different data sources require different processing pipelines
- Some paths involve more complex transformations than others
- You need to synchronize results from paths of varying complexity

In [9]:
# ============================================================================
# EXECUTE THE ASYMMETRIC WORKFLOW
# ============================================================================
# Watch the execution order carefully:
# 1. 'a' runs first
# 2. 'b' and 'c' start in parallel
# 3. 'x' runs after 'b' completes (but 'c' may still be running or already done)
# 4. 'd' runs only after BOTH 'x' AND 'c' complete

start = time.perf_counter()
result = graph.invoke({"state": []})
elapsed = time.perf_counter() - start

print("\n" + "="*50)
print("FINAL STATE:")
print("="*50)
print(f"Execution sequence: {result['state']}")
print(f"⏱️  Wall time: {elapsed:.2f}s — paced by the longer path (A, B, X, D), not the sum")
print("\nNote: C may appear before or after B/X depending on execution timing")

Adding step A to []
Adding step B to ['step A']
Adding step C to ['step A']
Adding step X to ['step A', 'step B', 'step C']
Adding step D to ['step A', 'step B', 'step C', 'step X']

FINAL STATE:
Execution sequence: ['step A', 'step B', 'step C', 'step X', 'step D']
⏱️  Wall time: 2.02s — paced by the longer path (A, B, X, D), not the sum

Note: C may appear before or after B/X depending on execution timing


---

## 🌐 Part 4: Real-World Example - Parallel Search with LLM Synthesis

Now let's apply what we've learned to a **practical use case**! We'll build an agent that:

1. **Searches Wikipedia** and **Web** in parallel (fan-out)
2. **Combines the results** from both sources (reducer)
3. **Generates an answer** using an LLM (fan-in)

```
            ┌──→ Wikipedia Search ──┐
START ──────┤                       ├──→ Generate Answer → END
            └──→ Web Search ────────┘
```

### Why This Pattern is Powerful

| Without Parallelization | With Parallelization |
|------------------------|---------------------|
| Wikipedia Search: 2s | Wikipedia Search: 2s |
| Web Search: 3s | Web Search: 3s (runs simultaneously) |
| **Total: 5s** | **Total: 3s** (40% faster!) |

The state schema holds the question, the answer, and a `context` list that both search nodes append to — the same reducer idea as Parts 1–3, now with real data.

In [10]:
# ============================================================================
# STATE SCHEMA FOR PARALLEL SEARCH AGENT
# ============================================================================

class State(TypedDict):
    """
    State schema for the parallel search agent.
    
    Fields:
    - question: The user's input question (read by search nodes)
    - answer: The LLM's final response (written by generate_answer node)
    - context: Search results from multiple sources (CRITICAL: uses reducer!)
    
    Why operator.add on context?
    ----------------------------
    Both search_web and search_wikipedia run IN PARALLEL and write to 'context'.
    Without the reducer, one would overwrite the other!
    With operator.add, their results are CONCATENATED:
        context from web + context from wikipedia = combined context
    """
    question: str                            # Input: User's question
    answer: str                              # Output: LLM's answer
    context: Annotated[list, operator.add]   # Accumulated search results

### 4.1 🔎 Node Functions

Now we define three node functions:

1. **`search_web`**: Searches the web with `TavilySearch` from `langchain-tavily` (runs in parallel)
2. **`search_wikipedia`**: Searches Wikipedia (runs in parallel)  
3. **`generate_answer`**: Uses LLM to synthesize an answer from combined context

Each function:
- Receives the current `state` as input
- Returns a dictionary with state updates
- For search nodes, wraps results in a list `[result]` to work with the `operator.add` reducer

In [11]:
# ============================================================================
# IMPORTS
# ============================================================================
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.document_loaders import WikipediaLoader
from langchain_tavily import TavilySearch

import wikipedia

# Wikimedia answers HTTP 429 to clients sending the wikipedia package's shared default
# User-Agent; its API etiquette asks for a descriptive one, which is not rate-limited.
wikipedia.set_user_agent(
    "ai-engineering-demystified-course/1.0 "
    "(https://github.com/Sourav692/ai-engineering-demystified-06-agent-fundamentals)"
)

# ============================================================================
# NODE FUNCTION 1: WEB SEARCH (Parallel Branch 1)
# ============================================================================
def search_web(state):
    """
    Search the web using Tavily API.

    This node runs IN PARALLEL with search_wikipedia.

    Args:
        state: Current graph state containing the user's question

    Returns:
        Dict with context update. Note the [formatted_search_docs] -
        wrapped in a list to work with the operator.add reducer!
    """
    print('--Getting Info from the Web--')

    # Initialize Tavily search tool (retrieves top 3 results)
    tavily_search = TavilySearch(max_results=3)

    # Perform the search; TavilySearch returns a dict with a "results" list
    search_docs = tavily_search.invoke({"query": state["question"]})["results"]

    # Format results into a readable string with source URLs
    # This helps the LLM understand where information came from
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'''Content: {doc['content']}\nSource: {doc['url']}'''
            for doc in search_docs
        ]
    )

    # IMPORTANT: Return as [list] to work with operator.add reducer
    # This allows parallel results to be concatenated, not overwritten
    return {"context": [formatted_search_docs]}


# ============================================================================
# NODE FUNCTION 2: WIKIPEDIA SEARCH (Parallel Branch 2)
# ============================================================================
def search_wikipedia(state):
    """
    Search Wikipedia for relevant articles.

    This node runs IN PARALLEL with search_web.

    Args:
        state: Current graph state containing the user's question

    Returns:
        Dict with context update wrapped in a list for the reducer
    """
    print('--Getting Info from the Wikipedia--')

    # Load up to 2 Wikipedia articles matching the query
    search_docs = WikipediaLoader(
        query=state['question'],
        load_max_docs=2
    ).load()

    # Format results with content and source
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'''Content: {doc.page_content}\nSource: {doc.metadata["source"]}'''
            for doc in search_docs
        ]
    )

    # Return as list for reducer compatibility
    return {"context": [formatted_search_docs]}


# ============================================================================
# NODE FUNCTION 3: ANSWER GENERATION (Fan-In Point)
# ============================================================================
def generate_answer(state):
    """
    Generate a final answer using the LLM.

    This node runs AFTER both search nodes complete (fan-in).
    It receives the COMBINED context from both Wikipedia and Web searches.

    Args:
        state: Graph state with accumulated context from parallel searches

    Returns:
        Dict with the LLM's answer
    """
    print('--Generating final answer--')

    # Extract accumulated context and original question from state
    # Note: context is a LIST of search results (thanks to our reducer)
    context = state["context"]
    question = state["question"]

    # Create a prompt template for the LLM
    # The LLM will use BOTH web and Wikipedia results to answer
    answer_template = """Answer the question {question}
                         using these context documents: {context}"""

    answer_instructions = answer_template.format(
        question=question,
        context=context  # Contains results from BOTH searches
    )

    # Invoke the LLM with system instructions and a human message
    answer = llm.invoke([
        SystemMessage(content=answer_instructions),
        HumanMessage(content="Answer the question.")
    ])

    # Return the answer text (no list needed - only one node writes to 'answer')
    return {"answer": answer.content}


print("✅ 2 search nodes and 1 answer node defined")

✅ 2 search nodes and 1 answer node defined


### 4.2 🏗️ Build the Agent Graph

`START` gets two outgoing edges (fan-out) and `generate_answer` gets two incoming edges (fan-in), so both searches run in the same step and the answer waits for both.

In [12]:
# ============================================================================
# BUILD THE PARALLEL SEARCH AGENT GRAPH
# ============================================================================
builder = StateGraph(State)

# Add all three nodes to the graph
builder.add_node("search_web", search_web)
builder.add_node("search_wikipedia", search_wikipedia)
builder.add_node("generate_answer", generate_answer)

# ============================================================================
# DEFINE PARALLEL FLOW
# ============================================================================

# FAN-OUT: Both searches start immediately from START
# Notice: START has TWO outgoing edges - this triggers parallel execution!
builder.add_edge(START, "search_wikipedia")  # Branch 1: Wikipedia
builder.add_edge(START, "search_web")        # Branch 2: Web (runs in parallel!)

# FAN-IN: Both searches converge at generate_answer
# LangGraph waits for BOTH to complete before running generate_answer
builder.add_edge("search_wikipedia", "generate_answer")
builder.add_edge("search_web", "generate_answer")

# Final step: Answer generation leads to END
builder.add_edge("generate_answer", END)

# Compile the graph
graph = builder.compile()

# ============================================================================
# VISUALIZE THE AGENT
# ============================================================================
print("✅ Parallel search graph compiled — both searches run simultaneously, then one LLM call")

✅ Parallel search graph compiled — both searches run simultaneously, then one LLM call


In [13]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = graph.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

🧩 Nodes: ['search_web', 'search_wikipedia', 'generate_answer']
🔗 Edges:
            __start__ ──▶ search_web         (fixed)
            __start__ ──▶ search_wikipedia   (fixed)
           search_web ──▶ generate_answer    (fixed)
     search_wikipedia ──▶ generate_answer    (fixed)
      generate_answer ──▶ __end__            (fixed)


### 4.3 🧪 Test the Parallel Search Agent

Let's test our agent with a real question! Watch the output to see:
1. Both search functions start (nearly) simultaneously
2. Results are combined before answer generation
3. The LLM synthesizes information from BOTH sources

In [14]:
# ============================================================================
# INVOKE THE PARALLEL SEARCH AGENT
# ============================================================================
# Ask a question that benefits from multiple sources:
# - Wikipedia: Provides foundational/historical information
# - Web Search: Provides recent news and current information

question = "Tell me about Nvidia and its growth"

print(f"Question: {question}")
print("="*50)
print()

# Execute the agent
# The graph will:
# 1. Run search_web AND search_wikipedia IN PARALLEL
# 2. Wait for both to complete
# 3. Pass combined context to generate_answer
# 4. Return the final state

start = time.perf_counter()
result = graph.invoke({"question": question})
elapsed = time.perf_counter() - start

print()
print("="*50)
print(f"✅ Agent execution complete in {elapsed:.1f}s")

Question: Tell me about Nvidia and its growth

--Getting Info from the Web--
--Getting Info from the Wikipedia--
--Generating final answer--

✅ Agent execution complete in 11.1s


In [15]:
# ============================================================================
# DISPLAY THE FINAL ANSWER
# ============================================================================
# The answer is generated by the LLM using context from BOTH:
# - Wikipedia articles (foundational knowledge)
# - Web search results (recent information)

print("="*60)
print("AGENT'S ANSWER")
print("="*60)
print()
print(result['answer'])

# Optional: Inspect the combined context that was passed to the LLM
print()
print("="*60)
print("SOURCES USED (Context)")
print("="*60)
print(f"Number of context chunks: {len(result['context'])}")
for i, ctx in enumerate(result['context']):
    print(f"\n--- Source {i+1} ---")
    print(ctx[:500] + "..." if len(ctx) > 500 else ctx)

AGENT'S ANSWER

Nvidia has experienced significant growth, primarily driven by its data center segment, which has been bolstered by the ongoing AI boom. In fiscal Q4, Nvidia reported a remarkable 75% year-over-year increase in data center revenue, reaching $62.3 billion. This surge is indicative of the growing demand for AI technologies, particularly agentic AI, which automates processes with minimal human oversight. 

In the fiscal year 2023-2024, Nvidia's revenue grew by an impressive 126% year-on-year, outpacing competitors like Intel and AMD, which saw declines in revenue. This growth is supported by Nvidia's strategic initiatives, including rapid product development, market expansion, and strong partnerships, solidifying its leadership in the AI and semiconductor industries.

Under the leadership of co-founder and CEO Jensen Huang, Nvidia has transformed from a gaming-focused company into a powerhouse in AI and high-performance computing. Huang's vision has led to the company's re

---
## 📝 Summary

In this notebook, we learned:

### 1. The sequential baseline
- A linear graph runs one node per step; wall time is the sum of the steps
- `Annotated[list, operator.add]` makes each node's update *append* instead of overwrite

### 2. Fan-out and fan-in
- Multiple edges out of one node run the targets in the same step, concurrently; multiple edges into one node make it wait for all of them
- Parallel nodes writing the same key need a reducer, or the last write wins and data is lost

### 3. Asymmetric parallel paths
- `builder.add_edge(["x", "c"], "d")` is an explicit sync point for branches of different lengths
- Wall time follows the longest path, not the sum

### 4. Multi-source retrieval
- Fan out at `START` to independent sources, fan in to one synthesis call: latency ≈ the slowest source
- Label or keep sources separate in the context so the LLM can use (and cite) both

| Pattern | Use Case | Implementation |
|---------|----------|----------------|
| Parallel Search | Multiple data sources | Fan-out at START, fan-in before synthesis |
| Sectioning / Voting | Independent subtasks, or the same check several times | `02_parallelization_sectioning_and_voting.ipynb` |
| Dynamic fan-out | The number of branches depends on the input | orchestrator-worker, `modules/08_orchestrator_worker` |

### Next Steps
- **`02_parallelization_sectioning_and_voting.ipynb`**: the pattern's two LLM variants — split a task into sections, or run the same check several times and vote